# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** Regulation (EU) 2016/679 – General Data Protection Regulation (GDPR), 88 pages.
It is a long legal text full of general rules, exceptions ("shall not apply where…") and prohibitions,
which makes it a good stress test for retrieval.

**Pipelines compared over the exact same chunk corpus**

| | Pipeline 1 – Baseline | Pipeline 2 – Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder retrieves Top-10 by cosine similarity | same Top-10 candidates |
| Stage 2 | – | Cross-encoder scores all 10 (query, chunk) pairs, re-sorts |
| LLM context | Top-3 from bi-encoder | Top-3 after re-ranking |
| Prompt | identical | identical |

## 1. Setup & configuration

In [1]:
import hashlib
import json
import os
import platform
import re
import time
from dataclasses import dataclass, field
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"  # harmless Windows cache warning

import numpy as np
import pandas as pd
import pymupdf
from sentence_transformers import CrossEncoder, SentenceTransformer

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

In [2]:
PDF_PATH = Path("data/gdpr.pdf")

BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

LLM_MODEL = "gemini-3.6-flash"
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")  # set this before starting Jupyter

MAX_CHUNK_WORDS = 180  # upper bound on chunk length
RETRIEVE_K = 10        # candidates returned by the bi-encoder
CONTEXT_K = 3          # chunks passed to the LLM

## 2. Load and clean the PDF

Every page ends with the Official Journal footer (date, `L 119/NN`, journal name, `EN`) and some pages carry
legal footnotes such as `(1) OJ C 229, …`. Both are noise for retrieval, so they are removed here.
Each remaining line keeps its page number so that chunks can be traced back to the source.

In [3]:
FOOTER_LINE = re.compile(r"^(\d{1,2}\.\d{1,2}\.\d{4}|L \d+/\d+|Official Journal of the European Union|EN)$")
FOOTNOTE_START = re.compile(r"^\(\d\) (Regulation|Directive|Decision|Commission|Council|Position|OJ)")


def read_pdf_lines(path):
    """Return a list of (page_number, line_text) with page footers removed."""
    lines = []
    for page in pymupdf.open(path):
        for line in page.get_text().splitlines():
            line = line.strip()
            if line and not FOOTER_LINE.match(line):
                lines.append((page.number + 1, line))
    return lines


pdf_lines = read_pdf_lines(PDF_PATH)
print(f"{len(pdf_lines)} lines of text extracted")

4187 lines of text extracted


## 3. Structure-aware chunking

The GDPR has two parts:

* **Recitals (1)–(173)** – explanatory paragraphs before the words *"HAVE ADOPTED THIS REGULATION"*.
* **Articles 1–99** – the binding rules, each starting with `Article N` followed by its title.

Chunks never cross a recital/article boundary. Inside an article, text is split on numbered paragraphs
(`1.`, `2.`, …) and paragraphs are packed together until `MAX_CHUNK_WORDS` is reached.
Each chunk is prefixed with its section label (e.g. `[Article 34 - Communication of a personal data breach …]`)
so both encoders know where the text comes from.

In [4]:
@dataclass
class Chunk:
    chunk_id: int
    section: str
    page: int
    text: str


def is_recital_header(match, last_recital):
    """Recitals are numbered in order; (1)-(99) stand alone on a line, (100)+ share it with text."""
    number, rest = int(match.group(1)), match.group(2)
    if number != last_recital + 1:
        return False
    return number >= 100 or not rest


def split_into_sections(lines):
    """Group lines into recitals and articles."""
    sections = []
    current = None
    in_articles = False
    last_recital = 0
    footnote_page = None
    i = 0

    while i < len(lines):
        page, line = lines[i]
        recital = re.fullmatch(r"\((\d{1,3})\)\s*(.*)", line)
        article = re.fullmatch(r"Article (\d{1,2})", line)

        if line.startswith("HAVE ADOPTED THIS REGULATION"):
            in_articles = True
        elif FOOTNOTE_START.match(line):
            footnote_page = page  # footnotes run to the bottom of the page
        elif page == footnote_page:
            pass
        elif not in_articles and recital and is_recital_header(recital, last_recital):
            last_recital += 1
            first_line = [recital.group(2)] if recital.group(2) else []
            current = {"section": f"Recital {last_recital}", "page": page, "lines": first_line}
            sections.append(current)
        elif in_articles and article:
            title = lines[i + 1][1]
            current = {"section": f"Article {article.group(1)} - {title}", "page": page, "lines": []}
            sections.append(current)
            i += 1  # the title line is consumed
        elif in_articles and re.fullmatch(r"(CHAPTER|Section) [IVX\d]+", line):
            i += 1  # skip chapter/section headings and their titles
        elif current is not None:
            current["lines"].append(line)
        i += 1

    return sections


def split_paragraphs(section_lines):
    """Split an article on its numbered paragraphs ("1.", "2.", ...)."""
    paragraphs, buffer = [], []
    for line in section_lines:
        if re.fullmatch(r"\d{1,2}\.", line) and buffer:
            paragraphs.append(" ".join(buffer))
            buffer = []
        buffer.append(line)
    if buffer:
        paragraphs.append(" ".join(buffer))
    return paragraphs


def pack_paragraphs(paragraphs, max_words):
    """Greedily merge paragraphs into chunks of at most max_words words."""
    chunks, current = [], []
    for paragraph in paragraphs:
        words = paragraph.split()
        while len(words) > max_words:  # a single very long paragraph
            if current:
                chunks.append(" ".join(current))
                current = []
            chunks.append(" ".join(words[:max_words]))
            words = words[max_words:]
        if current and len(current) + len(words) > max_words:
            chunks.append(" ".join(current))
            current = []
        current.extend(words)
    if current:
        chunks.append(" ".join(current))
    return chunks


def build_chunks(lines, max_words=MAX_CHUNK_WORDS):
    chunks = []
    for section in split_into_sections(lines):
        for body in pack_paragraphs(split_paragraphs(section["lines"]), max_words):
            text = f"[{section['section']}] {body}"
            chunks.append(Chunk(len(chunks), section["section"], section["page"], text))
    return chunks


chunks = build_chunks(pdf_lines)
chunk_texts = [c.text for c in chunks]

word_counts = [len(t.split()) for t in chunk_texts]
n_articles = len({c.section for c in chunks if c.section.startswith("Article")})
n_recitals = len({c.section for c in chunks if c.section.startswith("Recital")})
print(f"{len(chunks)} chunks | {n_recitals} recitals | {n_articles} articles")
print(f"words per chunk: mean={np.mean(word_counts):.0f}, max={max(word_counts)}")

459 chunks | 173 recitals | 99 articles
words per chunk: mean=121, max=198


In [5]:
# Example: Article 34 is split into its general rule (para 1-2) and its exceptions (para 3)
for c in chunks:
    if c.section.startswith("Article 34 "):
        print(f"chunk {c.chunk_id} (page {c.page}):\n{c.text}\n")

chunk 304 (page 52):
[Article 34 - Communication of a personal data breach to the data subject] 1. When the personal data breach is likely to result in a high risk to the rights and freedoms of natural persons, the controller shall communicate the personal data breach to the data subject without undue delay. 2. The communication to the data subject referred to in paragraph 1 of this Article shall describe in clear and plain language the nature of the personal data breach and contain at least the information and measures referred to in points (b), (c) and (d) of Article 33(3).

chunk 305 (page 52):
[Article 34 - Communication of a personal data breach to the data subject] 3. The communication to the data subject referred to in paragraph 1 shall not be required if any of the following conditions are met: (a) the controller has implemented appropriate technical and organisational protection measures, and those measures were applied to the personal data affected by the personal data breach

## 4. Vector store with a Bi-Encoder (Pipeline 1, step a)

The bi-encoder embeds every chunk **independently** into a 384-dim vector. Embeddings are L2-normalised,
so the inner product equals cosine similarity. With a few hundred chunks a NumPy matrix is a perfectly good
exact vector store (a FAISS `IndexFlatIP` would compute exactly the same thing).

In [6]:
class VectorStore:
    """Minimal exact-search vector store using normalised embeddings and inner product."""

    def __init__(self, model):
        self.model = model
        self.embeddings = None

    def index(self, texts):
        self.embeddings = self.model.encode(texts, batch_size=32, normalize_embeddings=True)

    def search(self, query, k):
        """Return [(chunk_index, cosine_score)] for the k most similar chunks."""
        query_vec = self.model.encode(query, normalize_embeddings=True)
        scores = self.embeddings @ query_vec
        top = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in top]


bi_encoder = SentenceTransformer(BI_ENCODER_NAME)
store = VectorStore(bi_encoder)

start = time.perf_counter()
store.index(chunk_texts)
index_seconds = time.perf_counter() - start
print(f"Indexed {len(chunk_texts)} chunks -> matrix {store.embeddings.shape} in {index_seconds:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 459 chunks -> matrix (459, 384) in 49.21s


## 5. Cross-Encoder (Pipeline 2, step b)

The cross-encoder reads the query and the chunk **together** in one transformer pass and outputs a single
relevance score. It cannot pre-compute anything, so it is only applied to the 10 candidates.

In [7]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 6. LLM and the shared prompt

Both pipelines use **exactly the same prompt template**; only the three context chunks differ.
Temperature is 0 so differences in the answers come from the context, not from sampling.
Generated answers are cached in `data/llm_answers.json`, so re-running the notebook reuses them
(and works without an API key). If no key and no cached answer exist, the answer cells show a placeholder.

In [8]:
PROMPT_TEMPLATE = """You are a legal assistant answering questions about the EU General Data Protection Regulation (GDPR).
Answer the question using ONLY the context below. Cite the Article or Recital you rely on.
If the context does not contain the answer, say "The provided context does not answer this."

Context:
{context}

Question: {question}

Answer:"""


def build_prompt(question, hits):
    context = "\n\n".join(f"({i}) {hit.chunk.text}" for i, hit in enumerate(hits, start=1))
    return PROMPT_TEMPLATE.format(context=context, question=question)


llm_client = None
if GEMINI_API_KEY:
    from google import genai
    from google.genai import errors, types

    llm_client = genai.Client(api_key=GEMINI_API_KEY)
    llm_config = types.GenerateContentConfig(temperature=0.0)
else:
    print("GEMINI_API_KEY not set - using cached answers from data/llm_answers.json only.")


# Answers are cached on disk so re-running the notebook does not spend API quota again.
LLM_CACHE_PATH = Path("data/llm_answers.json")
llm_cache = json.loads(LLM_CACHE_PATH.read_text(encoding="utf-8")) if LLM_CACHE_PATH.exists() else {}

RETRYABLE_STATUS = {429, 500, 503}  # rate limit / temporary server errors


def call_llm(prompt, max_retries=6):
    """Call Gemini once, waiting and retrying on rate limits or temporary outages."""
    for attempt in range(max_retries):
        try:
            start = time.perf_counter()
            response = llm_client.models.generate_content(model=LLM_MODEL, contents=prompt, config=llm_config)
            return response.text.strip(), (time.perf_counter() - start) * 1000
        except errors.APIError as error:
            if error.code not in RETRYABLE_STATUS or attempt == max_retries - 1:
                raise
            time.sleep(30)


def generate_answer(prompt):
    """Return (answer_text, latency_ms of the API call)."""
    key = hashlib.sha256(f"{LLM_MODEL}|{prompt}".encode()).hexdigest()
    if key in llm_cache:
        return llm_cache[key]["answer"], llm_cache[key]["latency_ms"]
    if llm_client is None:
        return "[LLM skipped - set GEMINI_API_KEY to generate answers]", None

    answer, latency_ms = call_llm(prompt)
    llm_cache[key] = {"answer": answer, "latency_ms": latency_ms}
    LLM_CACHE_PATH.write_text(json.dumps(llm_cache, indent=2), encoding="utf-8")
    return answer, latency_ms

GEMINI_API_KEY not set - using cached answers from data/llm_answers.json only.


## 7. The two retrieval pipelines

In [9]:
@dataclass
class Hit:
    chunk: Chunk
    baseline_rank: int          # 1-based rank from the bi-encoder
    bi_score: float             # cosine similarity
    rerank_rank: int = None     # 1-based rank after the cross-encoder
    ce_score: float = None      # cross-encoder relevance logit


def baseline_retrieve(query, k=RETRIEVE_K):
    """Pipeline 1 (a-b): Top-k chunks by cosine similarity."""
    results = store.search(query, k)
    return [Hit(chunks[idx], rank, score) for rank, (idx, score) in enumerate(results, start=1)]


def rerank(query, candidates):
    """Pipeline 2 (b-c): score every (query, chunk) pair and sort by cross-encoder score."""
    scores = cross_encoder.predict([(query, hit.chunk.text) for hit in candidates])
    for hit, score in zip(candidates, scores):
        hit.ce_score = float(score)
    reranked = sorted(candidates, key=lambda hit: hit.ce_score, reverse=True)
    for rank, hit in enumerate(reranked, start=1):
        hit.rerank_rank = rank
    return reranked


@dataclass
class PipelineRun:
    query: str
    baseline_hits: list           # Top-10 in bi-encoder order
    reranked_hits: list           # same 10 in cross-encoder order
    retrieval_ms: float           # bi-encoder search time
    rerank_ms: float              # cross-encoder time
    answer_1: str = None
    answer_2: str = None
    llm_1_ms: float = None
    llm_2_ms: float = None


def run_pipelines(query, with_llm=True):
    """Run both pipelines on the same Top-10 candidate set."""
    t0 = time.perf_counter()
    candidates = baseline_retrieve(query)
    t1 = time.perf_counter()
    reranked = rerank(query, list(candidates))
    t2 = time.perf_counter()

    run = PipelineRun(query, candidates, reranked, (t1 - t0) * 1000, (t2 - t1) * 1000)

    if with_llm:
        run.answer_1, run.llm_1_ms = generate_answer(build_prompt(query, candidates[:CONTEXT_K]))
        run.answer_2, run.llm_2_ms = generate_answer(build_prompt(query, reranked[:CONTEXT_K]))
    return run

## 8. Test cases (failure-mode stress tests)

Each test case lists **evidence phrases** taken verbatim from the GDPR text that a correct answer needs.
A chunk counts as the *critical chunk* if it contains one of these phrases. This makes the rank shifts
measurable instead of eyeballed.

In [10]:
@dataclass
class TestCase:
    name: str
    category: str
    query: str
    evidence: list = field(default_factory=list)
    why_hard: str = ""


TEST_CASES = [
    TestCase(
        name='Data portability vs. international transfers',
        category='Keyword trap',
        query='Can I request a transfer of my personal data to another controller?',
        evidence=['right to have the personal data transmitted directly from one controller to another'],
        why_hard="'transfer' is the key term of Chapter V (transfers to third countries); the real answer (Art. 20 data portability) says 'transmit' instead.",
    ),
    TestCase(
        name='Sub-processor liability',
        category='Keyword trap',
        query='Who is liable if a sub-processor breaches its obligations?',
        evidence=['the initial processor shall remain fully liable to the controller'],
        why_hard="'liable' and 'breach' match the general compensation rule (Art. 82); the specific sub-processor rule is hidden in Art. 28(4).",
    ),
    TestCase(
        name='Telling individuals about a breach',
        category='Exception / corner case',
        query='Do I always have to tell affected individuals about a data breach?',
        evidence=['shall not be required if any of the following conditions'],
        why_hard='The general duty (Art. 34(1)) is a near-perfect match; the exceptions (encryption, disproportionate effort) sit in a separate chunk, Art. 34(3).',
    ),
    TestCase(
        name='Breach not reported to the authority',
        category='Exception / corner case',
        query='When does a data breach not need to be reported to the supervisory authority?',
        evidence=['unless the personal data breach is unlikely to result in a risk'],
        why_hard="The exception is a single 'unless …' clause inside Art. 33(1); many breach chunks look similar.",
    ),
    TestCase(
        name='Dismissal of the DPO',
        category='Negation / role limit',
        query='Can a data protection officer be fired for doing their job?',
        evidence=['shall not be dismissed or penalised'],
        why_hard="Every DPO article (37, 38, 39) is topically similar; only one paragraph contains the prohibition 'shall not be dismissed', and 'fired' never appears in the text.",
    ),
    TestCase(
        name='Police and criminal investigations',
        category='Document-specific (scope exclusion)',
        query='Does GDPR apply to the police investigating crimes?',
        evidence=['prevention, investigation, detection or prosecution of criminal offences'],
        why_hard='Articles about criminal-conviction data (Art. 10) look relevant, but the answer is the scope exclusion in Art. 2(2)(d) / Recital 19 (handled by Directive 2016/680 instead).',
    ),
    TestCase(
        name='Regulator ignores a complaint',
        category='Limitation: re-ranking hurts',
        query='What can I do if the regulator ignores my complaint?',
        evidence=['does not handle a complaint or does not inform the data subject within three months'],
        why_hard='Counter-example: the cross-encoder prefers recitals that talk about complaints in general and pushes the Art. 78(2) remedy down.',
    ),
    TestCase(
        name='Switching provider',
        category='Limitation: recall ceiling',
        query='Can I take my data with me when I switch to another provider?',
        evidence=['right to have the personal data transmitted directly from one controller to another'],
        why_hard='Counter-example: Art. 20 is ranked #11 by the bi-encoder, so it never reaches the re-ranker. Re-ranking can only reorder what stage 1 retrieved.',
    ),
]



def critical_rank(hits, evidence, rank_attr):
    """Rank of the first hit that contains any evidence phrase (None if not in the Top-10)."""
    for hit in hits:
        if any(phrase.lower() in hit.chunk.text.lower() for phrase in evidence):
            return getattr(hit, rank_attr)
    return None


def is_critical(hit, evidence):
    return any(phrase.lower() in hit.chunk.text.lower() for phrase in evidence)

### 8.1 Rank-shift summary

Retrieval only (no LLM yet). `baseline rank` is the position of the critical chunk in the bi-encoder Top-10,
`re-ranked rank` its position after the cross-encoder. Anything > 3 is **not seen by the LLM**.

In [11]:
runs = {case.name: run_pipelines(case.query, with_llm=False) for case in TEST_CASES}


def fmt_rank(rank):
    return f"#{rank}" if rank else "not in Top-10"


summary_rows = []
for case in TEST_CASES:
    run = runs[case.name]
    before = critical_rank(run.baseline_hits, case.evidence, "baseline_rank")
    after = critical_rank(run.reranked_hits, case.evidence, "rerank_rank")
    critical = next((h for h in run.reranked_hits if is_critical(h, case.evidence)), None)
    summary_rows.append({
        "case": case.name,
        "category": case.category,
        "critical chunk": f"{critical.chunk.chunk_id} ({critical.chunk.section.split(' - ')[0]})" if critical else "-",
        "baseline rank": fmt_rank(before),
        "re-ranked rank": fmt_rank(after),
        "in LLM context (P1 / P2)": f"{'yes' if before and before <= CONTEXT_K else 'NO'} / "
                                    f"{'yes' if after and after <= CONTEXT_K else 'NO'}",
    })

pd.DataFrame(summary_rows).set_index("case")

,category,critical chunk,baseline rank,re-ranked rank,in LLM context (P1 / P2)
case,,,,,
Data portability vs. international transfers,Keyword trap,274 (Article 20),#4,#2,NO / yes
Sub-processor liability,Keyword trap,292 (Article 28),#4,#1,NO / yes
Telling individuals about a breach,Exception / corner case,305 (Article 34),#6,#2,NO / yes
Breach not reported to the authority,Exception / corner case,302 (Article 33),#4,#1,NO / yes
Dismissal of the DPO,Negation / role limit,318 (Article 38),#6,#2,NO / yes
Police and criminal investigations,Document-specific (scope exclusion),20 (Recital 19),#4,#1,NO / yes
Regulator ignores a complaint,Limitation: re-ranking hurts,424 (Article 78),#4,#7,NO / NO
Switching provider,Limitation: recall ceiling,-,not in Top-10,not in Top-10,NO / NO


### 8.2 Detailed view per test case

For each case: the full Top-10 with both scores, then Answer 1 (baseline context) vs Answer 2 (re-ranked context).
`*` marks the critical chunk.

In [12]:
def rank_table(run, case):
    rows = []
    for hit in run.baseline_hits:
        rows.append({
            "chunk": hit.chunk.chunk_id,
            "section": hit.chunk.section[:60],
            "critical": "*" if is_critical(hit, case.evidence) else "",
            "bi-encoder rank": hit.baseline_rank,
            "cosine": round(hit.bi_score, 3),
            "cross-encoder rank": hit.rerank_rank,
            "CE score": round(hit.ce_score, 2),
            "shift": hit.baseline_rank - hit.rerank_rank,
        })
    return pd.DataFrame(rows).set_index("chunk").sort_values("cross-encoder rank")


def show_case(case):
    run = run_pipelines(case.query, with_llm=True)
    runs[case.name] = run
    before = critical_rank(run.baseline_hits, case.evidence, "baseline_rank")
    after = critical_rank(run.reranked_hits, case.evidence, "rerank_rank")

    print("=" * 110)
    print(f"{case.name}  [{case.category}]")
    print(f"Query: {case.query}")
    print(f"Why it is hard: {case.why_hard}")
    print(f"Critical chunk: {fmt_rank(before)} -> {fmt_rank(after)}")
    display(rank_table(run, case))
    print("--- Answer 1 (baseline Top-3:", [h.chunk.chunk_id for h in run.baseline_hits[:CONTEXT_K]], ")")
    print(run.answer_1)
    print()
    print("--- Answer 2 (re-ranked Top-3:", [h.chunk.chunk_id for h in run.reranked_hits[:CONTEXT_K]], ")")
    print(run.answer_2)

#### Case 1: Data portability vs. international transfers (Keyword trap)

In [13]:
show_case(TEST_CASES[0])

Data portability vs. international transfers  [Keyword trap]
Query: Can I request a transfer of my personal data to another controller?
Why it is hard: 'transfer' is the key term of Chapter V (transfers to third countries); the real answer (Art. 20 data portability) says 'transmit' instead.
Critical chunk: #4 -> #2


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
346,Article 46 - Transfers subject to appropriate safeguards,,9,0.511,1,5.23,8
274,Article 20 - Right to data portability,*,4,0.537,2,4.77,2
266,Article 15 - Right of access by the data subject,,7,0.521,3,4.14,4
357,Article 49 - Derogations for specific situations,,3,0.561,4,4.10,-1
85,Recital 68,,1,0.708,5,4.08,-4
259,Article 14 - Information to be provided where personal data,,8,0.519,6,3.16,2
78,Recital 64,,5,0.530,7,3.03,-2
146,Recital 113,,10,0.502,8,2.85,2
57,Recital 48,,2,0.609,9,2.45,-7


--- Answer 1 (baseline Top-3: [85, 57, 357] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [346, 274, 266] )
Yes. According to Article 20(2), when exercising your right to data portability, you have the right to have your personal data transmitted directly from one controller to another, where technically feasible. 

Under Article 20(1), this right applies where:
1. The processing is based on consent (pursuant to point (a) of Article 6(1) or point (a) of Article 9(2)) or on a contract (pursuant to point (b) of Article 6(1)); and 
2. The processing is carried out by automated means.

*(Citation: Article 20(1) and Article 20(2))*


#### Case 2: Sub-processor liability (Keyword trap)

In [14]:
show_case(TEST_CASES[1])

Sub-processor liability  [Keyword trap]
Query: Who is liable if a sub-processor breaches its obligations?
Why it is hard: 'liable' and 'breach' match the general compensation rule (Art. 82); the specific sub-processor rule is hidden in Art. 28(4).
Critical chunk: #4 -> #1


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
292,Article 28 - Processor,*,4,0.615,1,3.44,3
429,Article 82 - Right to compensation and liability,,1,0.695,2,2.54,-1
103,Recital 80,,7,0.531,3,1.81,4
188,Recital 146,,2,0.625,4,0.48,-2
430,Article 82 - Right to compensation and liability,,5,0.608,5,0.45,0
189,Recital 146,,3,0.625,6,-1.23,-3
122,Recital 95,,10,0.519,7,-2.00,3
294,Article 28 - Processor,,6,0.569,8,-3.30,-2
433,Article 83 - General conditions for imposing administrative,,9,0.521,9,-5.08,0


--- Answer 1 (baseline Top-3: [429, 188, 189] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [292, 429, 103] )
Based on the provided context, if a sub-processor ("another processor") fails to fulfill its data protection obligations, the **initial processor** remains fully liable to the controller for the performance of that sub-processor's obligations.

**Citation:** Article 28(4) [Article 28 - Processor]


#### Case 3: Telling individuals about a breach (Exception / corner case)

In [15]:
show_case(TEST_CASES[2])

Telling individuals about a breach  [Exception / corner case]
Query: Do I always have to tell affected individuals about a data breach?
Why it is hard: The general duty (Art. 34(1)) is a near-perfect match; the exceptions (encryption, disproportionate effort) sit in a separate chunk, Art. 34(3).
Critical chunk: #6 -> #2


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
109,Recital 85,,7,0.596,1,2.71,6
305,Article 34 - Communication of a personal data breach to the,*,6,0.599,2,1.30,4
111,Recital 86,,5,0.603,3,0.38,2
113,Recital 88,,3,0.614,4,0.01,-1
306,Article 34 - Communication of a personal data breach to the,,1,0.625,5,-0.45,-4
303,Article 33 - Notification of a personal data breach to the s,,2,0.624,6,-0.53,-4
112,Recital 87,,8,0.572,7,-0.58,1
304,Article 34 - Communication of a personal data breach to the,,4,0.608,8,-0.87,-4
302,Article 33 - Notification of a personal data breach to the s,,9,0.566,9,-1.18,0


--- Answer 1 (baseline Top-3: [306, 303, 113] )
Based on the provided context, you do not always have to communicate a personal data breach to affected individuals (data subjects). 

According to **Article 34(4)**, if you have not already communicated the breach to the data subject, the supervisory authority may require you to do so after considering the likelihood of the breach resulting in a high risk, or it may decide that any of the conditions referred to in paragraph 3 are met.

--- Answer 2 (re-ranked Top-3: [109, 305, 111] )
No, you do not always have to inform affected individuals (data subjects) about a personal data breach. 

According to **Recital 86**, communication to the data subject is required when a personal data breach is "likely to result in a high risk to the rights and freedoms of the natural person."

Furthermore, according to **Article 34(3)**, communication to the data subject is **not required** if any of the following conditions are met:
1. **(a)** The control

#### Case 4: Breach not reported to the authority (Exception / corner case)

In [16]:
show_case(TEST_CASES[3])

Breach not reported to the authority  [Exception / corner case]
Query: When does a data breach not need to be reported to the supervisory authority?
Why it is hard: The exception is a single 'unless …' clause inside Art. 33(1); many breach chunks look similar.
Critical chunk: #4 -> #1


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
302,Article 33 - Notification of a personal data breach to the s,*,4,0.668,1,6.88,3
306,Article 34 - Communication of a personal data breach to the,,2,0.701,2,6.66,0
109,Recital 85,,5,0.624,3,5.82,2
112,Recital 87,,1,0.702,4,4.67,-3
303,Article 33 - Notification of a personal data breach to the s,,3,0.688,5,4.12,-2
179,Recital 141,,9,0.586,6,2.97,3
111,Recital 86,,8,0.599,7,2.41,1
351,Article 47 - Binding corporate rules,,6,0.613,8,-0.51,-2
374,Article 58 - Powers,,10,0.580,9,-1.40,1


--- Answer 1 (baseline Top-3: [112, 306, 303] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [302, 306, 109] )
A personal data breach does not need to be reported to the supervisory authority if the breach is unlikely to result in a risk to the rights and freedoms of natural persons (Article 33(1)), provided the controller is able to demonstrate this in accordance with the accountability principle (Recital 85).


#### Case 5: Dismissal of the DPO (Negation / role limit)

In [17]:
show_case(TEST_CASES[4])

Dismissal of the DPO  [Negation / role limit]
Query: Can a data protection officer be fired for doing their job?
Why it is hard: Every DPO article (37, 38, 39) is topically similar; only one paragraph contains the prohibition 'shall not be dismissed', and 'fired' never appears in the text.
Critical chunk: #6 -> #2


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
319,Article 38 - Position of the data protection officer,,1,0.615,1,2.45,0
318,Article 38 - Position of the data protection officer,*,6,0.502,2,2.21,4
321,Article 39 - Tasks of the data protection officer,,3,0.528,3,1.79,0
320,Article 39 - Tasks of the data protection officer,,4,0.518,4,0.72,0
317,Article 37 - Designation of the data protection officer,,2,0.541,5,0.72,-3
316,Article 37 - Designation of the data protection officer,,5,0.505,6,-0.45,-1
112,Recital 87,,7,0.498,7,-6.22,0
306,Article 34 - Communication of a personal data breach to the,,8,0.490,8,-7.20,0
412,Article 70 - Tasks of the Board,,9,0.459,9,-7.86,0


--- Answer 1 (baseline Top-3: [319, 317, 321] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [319, 318, 321] )
Based on the provided context, no. A data protection officer shall not be dismissed or penalised by the controller or the processor for performing his or her tasks.

**Citation:** Article 38(3)


#### Case 6: Police and criminal investigations (Document-specific (scope exclusion))

In [18]:
show_case(TEST_CASES[5])

Police and criminal investigations  [Document-specific (scope exclusion)]
Query: Does GDPR apply to the police investigating crimes?
Why it is hard: Articles about criminal-conviction data (Art. 10) look relevant, but the answer is the scope exclusion in Art. 2(2)(d) / Recital 19 (handled by Directive 2016/680 instead).
Critical chunk: #4 -> #1


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
20,Recital 19,*,4,0.487,1,-8.28,3
222,Article 2 - Material scope,*,9,0.448,2,-9.21,7
249,Article 10 - Processing of personal data relating to crimina,,1,0.526,3,-10.39,-2
124,Recital 97,,8,0.460,4,-10.63,4
298,Article 30 - Records of processing activities,,3,0.504,5,-10.64,-2
316,Article 37 - Designation of the data protection officer,,6,0.470,6,-10.80,0
308,Article 35 - Data protection impact assessment,,7,0.468,7,-10.87,0
278,"Article 22 - Automated individual decision-making, including",,10,0.439,8,-11.06,2
239,Article 6 - Lawfulness of processing,,2,0.512,9,-11.13,-7


--- Answer 1 (baseline Top-3: [249, 239, 298] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [20, 222, 249] )
No. According to Article 2(2)(d) and Recital 19, the GDPR does not apply to the processing of personal data by competent authorities for the purposes of the prevention, investigation, detection, or prosecution of criminal offences.


#### Case 7: Regulator ignores a complaint (Limitation: re-ranking hurts)

In [19]:
show_case(TEST_CASES[6])

Regulator ignores a complaint  [Limitation: re-ranking hurts]
Query: What can I do if the regulator ignores my complaint?
Why it is hard: Counter-example: the cross-encoder prefers recitals that talk about complaints in general and pushes the Art. 78(2) remedy down.
Critical chunk: #4 -> #7


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
179,Recital 141,,5,0.352,1,-2.80,4
160,Recital 125,,3,0.371,2,-3.85,1
167,Recital 130,,1,0.382,3,-4.63,-2
402,Article 65 - Dispute resolution by the Board,,2,0.376,4,-6.37,-2
425,Article 79 - Right to an effective judicial remedy against a,,8,0.348,5,-6.75,3
382,Article 60 - Cooperation between the lead supervisory author,,6,0.351,6,-7.59,0
424,Article 78 - Right to an effective judicial remedy against a,*,4,0.366,7,-8.23,-3
313,Article 36 - Prior consultation,,9,0.348,8,-9.51,1
190,Recital 147,,7,0.349,9,-9.89,-2


--- Answer 1 (baseline Top-3: [167, 402, 160] )
The provided context does not answer this.

--- Answer 2 (re-ranked Top-3: [179, 160, 167] )
According to Recital 141, if the supervisory authority (regulator) does not act on your complaint, you have the right to an effective judicial remedy in accordance with Article 47 of the Charter. 

*(Source: Recital 141)*


#### Case 8: Switching provider (Limitation: recall ceiling)

In [20]:
show_case(TEST_CASES[7])

Switching provider  [Limitation: recall ceiling]
Query: Can I take my data with me when I switch to another provider?
Why it is hard: Counter-example: Art. 20 is ranked #11 by the bi-encoder, so it never reaches the re-ranker. Re-ranking can only reorder what stage 1 retrieved.
Critical chunk: not in Top-10 -> not in Top-10


,section,critical,bi-encoder rank,cosine,cross-encoder rank,CE score,shift
chunk,,,,,,,
339,Article 44 - General principle for transfers,,10,0.387,1,-6.64,9
85,Recital 68,,1,0.489,2,-7.75,-1
70,Recital 57,,2,0.435,3,-8.89,-1
346,Article 46 - Transfers subject to appropriate safeguards,,9,0.389,4,-8.93,5
266,Article 15 - Right of access by the data subject,,4,0.420,5,-9.14,-1
357,Article 49 - Derogations for specific situations,,6,0.408,6,-9.42,0
241,Article 7 - Conditions for consent,,8,0.392,7,-9.81,1
130,Recital 102,,3,0.429,8,-9.85,-5
57,Recital 48,,5,0.410,9,-9.95,-4


--- Answer 1 (baseline Top-3: [85, 70, 130] )
Based on the provided context, you can "have the personal data transmitted directly from one controller to another" [Recital 68].

--- Answer 2 (re-ranked Top-3: [339, 85, 70] )
The provided context does not answer this.


## 9. Latency profiling

Each query is run `N_REPEATS` times after a warm-up call. Retrieval stages are timed separately from LLM generation,
because LLM latency (timed per successful API call, excluding rate-limit waits) depends on the network/API and is the same order of magnitude for both pipelines.

In [21]:
import torch

print(f"Hardware: {platform.processor()} | CPU only | torch threads = {torch.get_num_threads()}")

N_REPEATS = 20

for case in TEST_CASES:  # warm-up (model loading, caches)
    run_pipelines(case.query, with_llm=False)

latency_rows = []
for case in TEST_CASES:
    for _ in range(N_REPEATS):
        run = run_pipelines(case.query, with_llm=False)
        latency_rows.append({"case": case.name, "bi-encoder search": run.retrieval_ms,
                             "cross-encoder re-rank": run.rerank_ms})

latency = pd.DataFrame(latency_rows)
latency["Pipeline 1 retrieval"] = latency["bi-encoder search"]
latency["Pipeline 2 retrieval"] = latency["bi-encoder search"] + latency["cross-encoder re-rank"]

retrieval_table = latency.drop(columns="case").describe(percentiles=[0.5, 0.95]).T[["mean", "50%", "95%", "std"]]
retrieval_table.columns = ["mean (ms)", "median (ms)", "p95 (ms)", "std (ms)"]
retrieval_table.round(2)

Hardware: Intel64 Family 6 Model 186 Stepping 2, GenuineIntel | CPU only | torch threads = 8


,mean (ms),median (ms),p95 (ms),std (ms)
bi-encoder search,42.33,46.33,81.38,27.25
cross-encoder re-rank,1165.31,1455.26,1737.29,594.41
Pipeline 1 retrieval,42.33,46.33,81.38,27.25
Pipeline 2 retrieval,1207.64,1502.90,1807.81,616.06


In [22]:
llm_rows = [{"case": name, "LLM call P1 (ms)": run.llm_1_ms, "LLM call P2 (ms)": run.llm_2_ms}
            for name, run in runs.items() if run.llm_1_ms is not None]

if llm_rows:
    llm_latency = pd.DataFrame(llm_rows).set_index("case")
    p1_total = retrieval_table.loc["Pipeline 1 retrieval", "mean (ms)"] + llm_latency["LLM call P1 (ms)"].mean()
    p2_total = retrieval_table.loc["Pipeline 2 retrieval", "mean (ms)"] + llm_latency["LLM call P2 (ms)"].mean()
    display(llm_latency.round(0))
    print(f"End-to-end mean: Pipeline 1 = {p1_total:.0f} ms, Pipeline 2 = {p2_total:.0f} ms")
else:
    print("LLM latency not measured (no API key).")

,LLM call P1 (ms),LLM call P2 (ms)
case,,
Data portability vs. international transfers,8387.0,4685.0
Sub-processor liability,3562.0,4226.0
Telling individuals about a breach,13425.0,6860.0
Breach not reported to the authority,4262.0,4776.0
Dismissal of the DPO,3526.0,22116.0
Police and criminal investigations,3029.0,5919.0
Regulator ignores a complaint,3814.0,3978.0
Switching provider,6884.0,4636.0


End-to-end mean: Pipeline 1 = 5903 ms, Pipeline 2 = 8357 ms


## 10. Scaling experiment: why not cross-encode the whole document?

Here the cross-encoder is run against **every** chunk for one query and compared to the bi-encoder search.
The bi-encoder's chunk embeddings are computed once offline; at query time it only embeds the query and does one
matrix-vector product. The cross-encoder has no offline part: every (query, chunk) pair is a full transformer pass.

In [23]:
query = TEST_CASES[0].query
pairs = [(query, text) for text in chunk_texts]

start = time.perf_counter()
store.search(query, RETRIEVE_K)
bi_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
cross_encoder.predict(pairs, batch_size=32)
ce_full_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
cross_encoder.predict(pairs[:RETRIEVE_K])
ce_top10_ms = (time.perf_counter() - start) * 1000

ce_per_pair_ms = ce_full_ms / len(pairs)
scaling = pd.DataFrame({
    "corpus size (chunks)": [len(chunks), 100_000, 10_000_000],
    "cross-encoder on every chunk (s)": [ce_full_ms / 1000, ce_per_pair_ms * 100_000 / 1000,
                                         ce_per_pair_ms * 10_000_000 / 1000],
}).set_index("corpus size (chunks)")

print(f"Bi-encoder search over {len(chunks)} chunks : {bi_ms:8.1f} ms")
print(f"Cross-encoder on Top-{RETRIEVE_K} only          : {ce_top10_ms:8.1f} ms")
print(f"Cross-encoder on all {len(chunks)} chunks      : {ce_full_ms:8.1f} ms  ({ce_per_pair_ms:.2f} ms per pair)")
scaling.round(1)

Bi-encoder search over 459 chunks :     50.0 ms
Cross-encoder on Top-10 only          :   1416.0 ms
Cross-encoder on all 459 chunks      :  57419.0 ms  (125.10 ms per pair)


,cross-encoder on every chunk (s)
corpus size (chunks),
459,57.4
100000,12509.6
10000000,1250958.3
